# Baseline Regression Model Development

This notebook compares Linear Regression, Decision Tree Regression, and Random Forest Regression for predicting the continuous `Exam_Score` target. Classification thresholds for High, Average, and Low performance are not defined, so no classification labels are created.

## 1. Import Libraries

Import the reusable training and evaluation helpers, along with the libraries needed for inspection and plotting.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import build_predictions_frame, plot_actual_vs_predicted, plot_model_comparison, plot_residuals
from preprocessing import identify_feature_types, load_data
from train import build_model_pipelines, split_features_and_target

sns.set_theme(style="whitegrid")

## 2. Load Dataset

The original CSV is loaded for model development. It is read only and is never overwritten.

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "StudentPerformanceFactors.csv"
df = load_data(data_path)
print(f"Dataset shape: {df.shape}")

## 3. Define Features and Target

`Exam_Score` remains a continuous numerical target. It is separated from all predictor features without arbitrary feature selection or target categorization.

In [ ]:
X, y = split_features_and_target(df)
numerical_columns, categorical_columns = identify_feature_types(df)

print("Target:", "Exam_Score")
print("Numerical predictors:", numerical_columns)
print("Categorical predictors:", categorical_columns)

## 4. Train-Test Split

The split is performed before fitting any preprocessing step. The test set is held out for baseline evaluation and is not used to fit or tune the models.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## 5. Build Model Pipelines

Each model receives its own preprocessing-plus-estimator pipeline. The preprocessing is unfitted at construction time and is fitted only when the pipeline receives `X_train`.

In [ ]:
pipelines = build_model_pipelines(numerical_columns, categorical_columns)
for model_name, pipeline in pipelines.items():
    print(model_name, "->", pipeline)


## 6. Train Linear Regression

Linear Regression provides a simple continuous-target baseline.

In [ ]:
linear_regression_pipeline = pipelines["Linear Regression"]
linear_regression_pipeline.fit(X_train, y_train)
linear_predictions = linear_regression_pipeline.predict(X_test)
print("Linear Regression trained successfully.")

## 7. Train Decision Tree Regressor

The Decision Tree baseline is configured with `random_state=42` for reproducibility.

In [ ]:
decision_tree_pipeline = pipelines["Decision Tree Regressor"]
decision_tree_pipeline.fit(X_train, y_train)
decision_tree_predictions = decision_tree_pipeline.predict(X_test)
print("Decision Tree Regressor trained successfully.")

## 8. Train Random Forest Regressor

The Random Forest baseline uses `random_state=42` and is not hyperparameter tuned.

In [ ]:
random_forest_pipeline = pipelines["Random Forest Regressor"]
random_forest_pipeline.fit(X_train, y_train)
random_forest_predictions = random_forest_pipeline.predict(X_test)
print("Random Forest Regressor trained successfully.")

## 9. Model Evaluation

All models are evaluated on the same held-out test set using MAE, MSE, RMSE, and RÃƒâ€šÃ‚Â².

In [ ]:
predictions = {
    "Linear Regression": linear_predictions,
    "Decision Tree Regressor": decision_tree_predictions,
    "Random Forest Regressor": random_forest_predictions,
}

from evaluate import calculate_regression_metrics

results = pd.DataFrame(
    [
        {"Model": name, **calculate_regression_metrics(y_test, prediction)}
        for name, prediction in predictions.items()
    ]
)
results

## 10. Model Comparison

The comparison table reports every requested metric. Lower MAE, MSE, and RMSE are better; higher RÃƒâ€šÃ‚Â² is better.

In [ ]:
results = results.sort_values("RMSE").reset_index(drop=True)
display(results)

results_path = PROJECT_ROOT / "outputs" / "results" / "model_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(results_path, index=False)
print(f"Saved model comparison results to {results_path}")

## 11. Actual vs Predicted Analysis

In [ ]:
predictions_frame = build_predictions_frame(y_test, predictions)
actual_vs_predicted_path = PROJECT_ROOT / "outputs" / "figures" / "actual_vs_predicted.png"
plot_actual_vs_predicted(predictions_frame, actual_vs_predicted_path)
print(f"Saved actual-vs-predicted plot to {actual_vs_predicted_path}")

## 12. Residual Analysis

Residuals are calculated as actual minus predicted scores. Patterns can indicate where a baseline model fits less accurately.

In [ ]:
residuals_path = PROJECT_ROOT / "outputs" / "figures" / "residual_analysis.png"
plot_residuals(predictions_frame, residuals_path)
print(f"Saved residual plot to {residuals_path}")

## 13. Feature Importance (where applicable)

Feature importance is model-specific. It is reported only if Random Forest is selected; Linear Regression and the un-tuned Decision Tree are not used to produce this optional section here.

In [ ]:
selected_model_name = results.iloc[0]["Model"]

if selected_model_name == "Random Forest Regressor":
    fitted_forest = pipelines[selected_model_name].named_steps["model"]
    fitted_preprocessor = pipelines[selected_model_name].named_steps["preprocessor"]
    feature_names = fitted_preprocessor.get_feature_names_out()
    feature_importance = (
        pd.Series(fitted_forest.feature_importances_, index=feature_names)
        .sort_values(ascending=False)
        .head(20)
    )
    display(feature_importance.to_frame(name="importance"))
    feature_importance.sort_values().plot(kind="barh", figsize=(10, 8))
    plt.title("Top Random Forest Feature Importances")
    plt.tight_layout()
    feature_importance_path = PROJECT_ROOT / "outputs" / "figures" / "feature_importance.png"
    plt.savefig(feature_importance_path, dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"{selected_model_name} was selected, so Random Forest feature importance is not applicable to the selected model.")

## 14. Final Baseline Model Selection

The baseline selection uses all four reported test metrics rather than a single metric. Lower MAE, MSE, and RMSE indicate smaller errors, while higher RÃƒâ€šÃ‚Â² indicates more explained variance. The model ranked first by RMSE is selected for the baseline artifact; the complete comparison table remains available for review.

In [ ]:
comparison_plot_path = PROJECT_ROOT / "outputs" / "figures" / "model_comparison.png"
plot_model_comparison(results, comparison_plot_path)

print("Selected baseline model:", selected_model_name)
print("Selection rationale: it has the lowest RMSE in the baseline comparison and is also evaluated using MAE, MSE, and R2 above.")

## 15. Classification Decision Note

`Exam_Score` is modeled as a continuous numerical target in this baseline stage. The project brief mentions High, Average, and Low performance, but it does not define score thresholds. No thresholds are invented, no classes are created, and no classifier is trained. Before adding classification, the project must define the thresholds, intended task, evaluation metric, and acceptable class balance.

## 16. Save Final Model

The selected fitted preprocessing-plus-model pipeline is saved as one joblib artifact for future prediction use. The raw CSV and human-readable cleaned CSV are not overwritten.

In [ ]:
model_path = PROJECT_ROOT / "models" / "best_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipelines[selected_model_name], model_path)

loaded_model = joblib.load(model_path)
reloaded_prediction = loaded_model.predict(X_test.iloc[[0]])

print(f"Saved model to {model_path}")
print("Reload test prediction:", reloaded_prediction[0])

## 17. Prediction and Recommendation Backend Validation

This small validation section uses the already saved `models/best_model.joblib`. It does not retrain the model, modify any CSV, or create classification labels.

In [ ]:
from predict import generate_student_prediction, load_model
from recommendations import RECOMMENDATION_THRESHOLDS

validation_examples = X_test.iloc[:3].copy()
for example_number, (_, student_row) in enumerate(validation_examples.iterrows(), start=1):
    result = generate_student_prediction(student_row.to_dict())
    print(f"Example {example_number}: {result}")

print("Recommendation thresholds derived from raw-data 25th percentiles:")
print(RECOMMENDATION_THRESHOLDS)

loaded_backend_model = load_model()
print("Backend model loaded:", type(loaded_backend_model).__name__)
print("Backend validation completed without retraining.")